# Personal AI Digital Twin — Phase 7 Demonstration

This notebook audits the **previously saved** model evaluation and the Phase 7 prototype evidence.

It does not train or fine-tune a model. The negotiation, SQLite persistence, privacy, and verification results shown here are read from saved experiment artifacts. The research benchmark metrics are reported separately from the simulated service-execution results.

Project root in the original Kaggle environment: `/kaggle/working/digital_twin`.


In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path("/kaggle/working/digital_twin")
if not ROOT.exists():
    ROOT = Path.cwd()

print("Project root:", ROOT)
assert ROOT.exists(), "Project root was not found. Set ROOT to the extracted project directory."


## 1. Artifact and reproducibility audit

In [ ]:
required = [
    "src/main.py",
    "src/digital_twin.py",
    "src/execution_verifier.py",
    "models/flan_t5_small_negotiation/corrected_fp32/config.json",
    "models/flan_t5_small_negotiation/corrected_fp32/model.safetensors",
    "experiments/flan_t5_small_negotiation/test_metrics_corrected.json",
    "experiments/flan_t5_small_negotiation/test_predictions_corrected.csv",
    "experiments/majority_target_baseline/metrics.json",
    "experiments/phase7_end_to_end/run_summary.json",
    "experiments/phase7_end_to_end/verification_report.json",
    "experiments/phase7_end_to_end/failure_path_trace.json",
    "PHASE7_ACADEMIC_REPORT_DRAFT.md",
]

audit = pd.DataFrame([
    {
        "artifact": rel,
        "exists": (ROOT / rel).is_file(),
        "size_bytes": (ROOT / rel).stat().st_size if (ROOT / rel).is_file() else None,
    }
    for rel in required
])
display(audit)
assert audit["exists"].all(), "One or more required artifacts are missing."
print("Artifact audit: PASS")


## 2. Saved FLAN-T5 test metrics

In [ ]:
metrics_path = ROOT / "experiments/flan_t5_small_negotiation/test_metrics_corrected.json"
metrics = json.loads(metrics_path.read_text())
print("Model:", metrics.get("model"))
print("Split:", metrics.get("split"))
print("Examples:", metrics.get("examples"))
print("Exact-match accuracy:", f"{metrics['exact_match_accuracy']:.2%}")
print("Category accuracy:", f"{metrics['category_accuracy']:.2%}")
print("Token micro-F1:", f"{metrics['token_micro_f1']:.2%}")

print("\nPer-category performance:")
display(pd.DataFrame(metrics["per_true_category"]).T)
print("\nTrue categories:")
display(pd.Series(metrics["true_category_counts"], name="examples"))
print("\nPredicted categories:")
display(pd.Series(metrics["predicted_category_counts"], name="predicted examples"))

print("Interpretation: category performance is imbalanced. In particular, the saved metrics show zero correct category predictions for disagreement and disconnect.")


## 3. Saved model versus majority-target baseline

In [ ]:
comparison_path = ROOT / "experiments/phase7_end_to_end/model_baseline_comparison.json"
comparison = json.loads(comparison_path.read_text())
display(pd.json_normalize(comparison).T.rename(columns={0: "value"}))
print("\nExact-match improvement is modest; category accuracy and minority-class results must be interpreted separately.")


## 4. Saved negotiation, execution, and verification evidence

In [ ]:
base = ROOT / "experiments/phase7_end_to_end"
summary = json.loads((base / "run_summary.json").read_text())
print(json.dumps(summary, indent=2))

steps = summary.get("steps", {})
assert summary.get("overall_status") == "passed"
assert steps.get("negotiation", {}).get("status") == "agreement"
assert steps.get("execution", {}).get("status") == "persisted"
assert steps.get("verification", {}).get("verified") is True
assert steps.get("privacy_policy", {}).get("passed") is True
assert steps.get("final_agreement_validation", {}).get("passed") is True
print("\nSaved successful-run assertions: PASS")


## 5. Failure-path and tamper-test evidence

In [ ]:
failure_path = json.loads((base / "failure_path_trace.json").read_text())
print("Failure-path evidence:")
print(json.dumps(failure_path, indent=2))

for candidate in [
    ROOT / "experiments/digital_twin_execution/tamper_detection_report.json",
    ROOT / "verification/tamper_detection_report.json",
]:
    if candidate.is_file():
        print("\nTamper-detection evidence:", candidate.relative_to(ROOT))
        print(candidate.read_text())


## 6. Prediction-file integrity

In [ ]:
pred_path = ROOT / "experiments/flan_t5_small_negotiation/test_predictions_corrected.csv"
predictions = pd.read_csv(pred_path)
print("Prediction rows:", len(predictions))
print("Columns:", predictions.columns.tolist())
assert len(predictions) == metrics["examples"] == 1052
assert {"true_category", "predicted_category", "exact_match"}.issubset(predictions.columns)
display(predictions.head(5))
print("\nPredicted-category distribution:")
display(predictions["predicted_category"].value_counts(dropna=False).to_frame("count"))


## 7. Conclusion and limitations

The saved evidence demonstrates a working rule-constrained negotiation prototype with simulated persistence, independent verification, and privacy checks. The benchmark experiment is a separate component: the model's exact-match accuracy is 15.11%, category accuracy is 84.60%, and token micro-F1 is 63.87% on the saved test split.

The saved category distribution reveals substantial class imbalance in predictions. The prototype's successful hotel-booking simulation must not be interpreted as proof that FLAN-T5 itself autonomously completed that booking; the demonstration uses the project's negotiation and verification logic.

No model training is performed in this notebook. See `PHASE7_ACADEMIC_REPORT_DRAFT.md` for the report draft; verify bibliography details and rerun the project tests in the intended submission environment before final submission.
